In [45]:
!pip install kaggle

In [46]:
import json
#username和key改为自己的kaggle的，如果不行，就可以直接用这个
token = {"username":"cskaoyan","key":"ff99d9d7ff71704e3e761217ceec03c5"}
with open('/content/kaggle.json', 'w') as file:
  json.dump(token, file)#json.dump类似于write

In [47]:
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle config set -n path -v /content

- path is now set to: /content


In [48]:
!kaggle datasets download -d slothkong/10-monkey-species

Dataset URL: https://www.kaggle.com/datasets/slothkong/10-monkey-species
License(s): CC0-1.0
User cancelled operation


In [49]:
!ls -lh datasets/slothkong/10-monkey-species/

total 548M
-rw-r--r-- 1 root root 548M Sep 26  2019 10-monkey-species.zip


In [50]:
!ls

data	  kaggle.json	     __pycache__  training    trainmodule_train.py
datasets  monkey_labels.txt  sample_data  validation


In [51]:
!pwd

/content


In [70]:
!unzip -o -d /content /content/datasets/slothkong/10-monkey-species/10-monkey-species.zip

Archive:  /content/datasets/slothkong/10-monkey-species/10-monkey-species.zip
  inflating: /content/monkey_labels.txt  
  inflating: /content/training/training/n0/n0018.jpg  
  inflating: /content/training/training/n0/n0019.jpg  
  inflating: /content/training/training/n0/n0020.jpg  
  inflating: /content/training/training/n0/n0021.jpg  
  inflating: /content/training/training/n0/n0022.jpg  
  inflating: /content/training/training/n0/n0023.jpg  
  inflating: /content/training/training/n0/n0024.jpg  
  inflating: /content/training/training/n0/n0025.jpg  
  inflating: /content/training/training/n0/n0026.jpg  
  inflating: /content/training/training/n0/n0027.jpg  
  inflating: /content/training/training/n0/n0028.jpg  
  inflating: /content/training/training/n0/n0029.jpg  
  inflating: /content/training/training/n0/n0030.jpg  
  inflating: /content/training/training/n0/n0031.jpg  
  inflating: /content/training/training/n0/n0032.jpg  
  inflating: /content/training/training/n0/n0033.jpg  


In [71]:
# 导入必要的库
import matplotlib as mpl
import matplotlib.pyplot as plt
# 在Jupyter notebook中内联显示图表
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm  # 进度条库
import torch
import torch.nn as nn
import torch.nn.functional as F

# 打印Python版本信息
print(sys.version_info)

# 打印各个库的版本信息
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)

# 设置设备：如果有GPU则使用GPU，否则使用CPU
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)


sys.version_info(major=3, minor=12, micro=12, releaselevel='final', serial=0)
matplotlib 3.10.0
numpy 2.0.2
pandas 2.2.2
sklearn 1.6.1
torch 2.9.0+cu126
cuda:0


In [72]:
!ls  /content/training/training

n0  n1	n2  n3	n4  n5	n6  n7	n8  n9


In [73]:
!ls /content/validation/

validation


# 数据预处理

In [74]:
# INSERT_YOUR_CODE
from torchvision import datasets, transforms
import os

# 设置数据目录
data_dir = '/content/'

# 定义预处理: resize到128x128, 转为tensor
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4363, 0.4328, 0.3291], std=[0.2427, 0.2382, 0.2413])
])

# 读取训练集和测试集
train_dataset = datasets.ImageFolder(root=os.path.join(data_dir, 'training/training'), transform=transform)
test_dataset = datasets.ImageFolder(root=os.path.join(data_dir, 'validation/validation'), transform=transform)

# 获取类别名（方便后续显示标签）
class_names = train_dataset.classes
class_names

['n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8', 'n9']

In [75]:
len(train_dataset)

1097

In [76]:
train_dataset[0][0].shape #特征

torch.Size([3, 128, 128])

In [77]:
train_dataset[0][1] #标签

0

In [78]:
len(test_dataset)

272

In [79]:
# INSERT_YOUR_CODE

# from torch.utils.data import DataLoader
# import torch

# loader = DataLoader(train_dataset, batch_size=64, shuffle=False, num_workers=2)

# mean = torch.zeros(3)
# std = torch.zeros(3)
# n_pixels = 0

# for images, _ in loader:  # images: [B, 3, 128, 128]
#     batch_pixels = images.numel() // 3  # total pixels per channel
#     mean += images.sum(dim=[0, 2, 3])
#     std  += (images ** 2).sum(dim=[0, 2, 3])
#     n_pixels += batch_pixels

# mean /= n_pixels
# std = torch.sqrt(std / n_pixels - mean ** 2)

# print("按通道均值:", mean)
# print("按通道标准差:", std)



In [80]:
from torch.utils.data import DataLoader

# 创建训练集和验证集的DataLoader
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,  # 训练时打乱数据
    num_workers=2  # 使用多进程加载数据
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,  # 测试时不需要打乱
    num_workers=2
)

print(f"训练集DataLoader批次数: {len(train_loader)}")
print(f"测试集DataLoader批次数: {len(test_loader)}")
print(f"每个批次大小: {batch_size}")

# 查看一个批次的数据
train_iter = iter(train_loader)
batch_images, batch_labels = next(train_iter)
print(f"批次图像张量形状: {batch_images.shape}")
print(f"批次标签张量形状: {batch_labels.shape}")
print(batch_labels)

训练集DataLoader批次数: 35
测试集DataLoader批次数: 9
每个批次大小: 32
批次图像张量形状: torch.Size([32, 3, 128, 128])
批次标签张量形状: torch.Size([32])
tensor([6, 4, 3, 8, 4, 4, 1, 7, 3, 8, 8, 2, 5, 2, 8, 0, 2, 5, 3, 7, 6, 3, 3, 8,
        9, 1, 9, 9, 8, 7, 5, 2])


# 搭建模型

In [90]:
import torch.nn as nn

class MonkeyNet(nn.Module):
    def __init__(self, num_classes=10):
        super(MonkeyNet, self).__init__()

        # 针对10-monkeys高分辨率彩色图片(3通道，128x128)，构建更深更宽更正则化的卷积网络
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),   # (B,32,128,128)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),  # (B,32,128,128)
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                       # (B,32,64,64)


            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),  # (B,64,64,64)
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),  # (B,64,64,64)
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                       # (B,64,32,32)


            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False), # (B,128,32,32)
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),# (B,128,32,32)
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                       # (B,128,16,16)


            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False), # (B,256,16,16)
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),# (B,256,16,16)
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2),                                     # (B,256,8,8)

        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*8*8, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 实例化模型
model = MonkeyNet(num_classes=10)


In [92]:
# 使用随机输入对模型进行一次前向计算以验证模型结构是否正确
import torch

dummy_input = torch.randn(32, 3, 128, 128)
output = model(dummy_input) #前向传播/前向计算/正向传播
print(f"Output shape: {output.shape}")


Output shape: torch.Size([32, 10])


In [93]:
# 输出model每一层的参数量
total_params = 0  # 初始化总参数量为0
print("各层参数量统计：")  # 打印参数统计表头
for name, param in model.named_parameters():  # 遍历模型中所有需要优化的参数
    if param.requires_grad:  # 只有需要梯度更新的参数才统计
        num_params = param.numel()  # 计算当前参数的元素总数
        total_params += num_params  # 更新总参数量
        print(f"{name}: {num_params}")  # 输出当前层的参数量
print(f"模型总参数量: {total_params}")  # 输出模型总参数量


各层参数量统计：
features.0.weight: 864
features.1.weight: 32
features.1.bias: 32
features.3.weight: 9216
features.4.weight: 32
features.4.bias: 32
features.7.weight: 18432
features.8.weight: 64
features.8.bias: 64
features.10.weight: 36864
features.11.weight: 64
features.11.bias: 64
features.14.weight: 73728
features.15.weight: 128
features.15.bias: 128
features.17.weight: 147456
features.18.weight: 128
features.18.bias: 128
features.21.weight: 294912
features.22.weight: 256
features.22.bias: 256
features.24.weight: 589824
features.25.weight: 256
features.25.bias: 256
classifier.1.weight: 2097152
classifier.1.bias: 128
classifier.3.weight: 1280
classifier.3.bias: 10
模型总参数量: 3271786


# 训练

In [95]:
import torch.nn as nn
import torch.optim as optim

# 初始化交叉熵损失函数，内部会做softmax
criterion = nn.CrossEntropyLoss()

# 初始化优化器（这里选用Adam，也可以使用SGD等）
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [96]:
import trainmodule_train
# 假设train_loader和val_loader已定义，device已经设为"cuda"或"cpu"
trainer = trainmodule_train.Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    eval_step=100
)

# 设定训练轮数
num_epochs = 20

# 开始训练
trainer.train(num_epochs)


Epoch [1/20]  Train Loss: 3.0125  Train Acc: 0.1404
Epoch [2/20]  Train Loss: 2.0358  Train Acc: 0.2407
[Step 100] Val Loss: 1.9978 Val Acc: 0.2831
Epoch [3/20]  Train Loss: 1.8629  Train Acc: 0.3300
Epoch [4/20]  Train Loss: 1.7221  Train Acc: 0.3965
Epoch [5/20]  Train Loss: 1.5116  Train Acc: 0.4585
[Step 200] Val Loss: 1.7517 Val Acc: 0.3897
Epoch [6/20]  Train Loss: 1.4261  Train Acc: 0.4786
Epoch [7/20]  Train Loss: 1.3347  Train Acc: 0.5369
Epoch [8/20]  Train Loss: 1.2202  Train Acc: 0.5734
[Step 300] Val Loss: 1.7492 Val Acc: 0.4412
Epoch [9/20]  Train Loss: 1.2776  Train Acc: 0.5397
Epoch [10/20]  Train Loss: 1.2068  Train Acc: 0.5542
Epoch [11/20]  Train Loss: 1.0021  Train Acc: 0.6354
[Step 400] Val Loss: 1.6526 Val Acc: 0.4596
Epoch [12/20]  Train Loss: 1.1486  Train Acc: 0.5980
Epoch [13/20]  Train Loss: 1.0213  Train Acc: 0.6345
Epoch [14/20]  Train Loss: 0.8392  Train Acc: 0.6782
[Step 500] Val Loss: 1.3777 Val Acc: 0.5000
Epoch [15/20]  Train Loss: 0.8236  Train Acc: 0

In [97]:
trainer.train(num_epochs)

Epoch [1/20]  Train Loss: 0.3891  Train Acc: 0.8706
Epoch [2/20]  Train Loss: 0.3964  Train Acc: 0.8578
[Step 100] Val Loss: 1.3580 Val Acc: 0.6213
Epoch [3/20]  Train Loss: 0.3182  Train Acc: 0.8897
Epoch [4/20]  Train Loss: 0.3142  Train Acc: 0.8961
Epoch [5/20]  Train Loss: 0.2537  Train Acc: 0.9180
[Step 200] Val Loss: 1.5104 Val Acc: 0.6103
Epoch [6/20]  Train Loss: 0.1846  Train Acc: 0.9417
Epoch [7/20]  Train Loss: 0.3168  Train Acc: 0.8824
Epoch [8/20]  Train Loss: 0.2028  Train Acc: 0.9307
[Step 300] Val Loss: 1.4376 Val Acc: 0.6140
Epoch [9/20]  Train Loss: 0.1108  Train Acc: 0.9681
Epoch [10/20]  Train Loss: 0.1601  Train Acc: 0.9480
Epoch [11/20]  Train Loss: 0.1293  Train Acc: 0.9608
[Step 400] Val Loss: 1.9286 Val Acc: 0.6287
Epoch [12/20]  Train Loss: 0.1439  Train Acc: 0.9553
Epoch [13/20]  Train Loss: 0.2315  Train Acc: 0.9161
Epoch [14/20]  Train Loss: 0.1780  Train Acc: 0.9426
[Step 500] Val Loss: 1.7645 Val Acc: 0.5662
Epoch [15/20]  Train Loss: 0.1404  Train Acc: 0